# Compute Jacobian-Vector products (JVP) and vector-Jacobian products (VJP)

Let $f\colon \mathbb{R}^n \rightarrow \mathbb{R}^m$ be multidimensional vector-valued function given by
$$
f(x) = (f_1(x), f_2(x)), \quad f_1(x) = 3 x_1^3 - x_2^2 + x_3, \quad f_2(x) = sin(x_1)x_2, 
$$
i.e., with $n=3$ and $m=2$. The _Jacobian_ $D_f \in \mathbb{R}^{m \times n}$ is defined as $D_f = \bigl(\partial_{x_j}f_i(x) \bigr)_{i, j}$ with
$$
\partial_{x_1}f_1(x) = 9 x_1^2, \quad 
\partial_{x_2}f_1(x) = - 2 x_2, \quad
\partial_{x_3}f_1(x) = 1, \quad
\partial_{x_1}f_2(x) = cos(x_1)x_2, \quad 
\partial_{x_2}f_2(x) = sin(x_1), \quad
\partial_{x_3}f_2(x) = 0 .
$$

TODO! Understand VJP and JVP

Import dependecies

In [1]:
import jax
from jax import numpy as jnp

create a random key

In [2]:
key = jax.random.PRNGKey(0)

Define multidimensional vector valued function $f$

In [3]:
def f(x):
    y = jnp.array([
        3 * x[0]**3 - x[1]**2 + x[2],
        jnp.sin(x[0])*x[1],
    ])
    return y

## Compute Jacobian-Vector Products (JVP)

1. Without batch

In [5]:
# set domain and codomain dimensions
n, m = 3, 2

# initialize input
key1, key2 = jax.random.split(key, 2)
x = jax.random.normal(key1, (n,))
v = jax.random.normal(key2, (n,))

# compute jvp
primals, tangents = jax.jvp(f, (x,), (v,))

#primals.shape, tangents.shape
x, v, f(x), primals, tangents

(Array([ 1.0040143, -0.9063372, -0.7481722], dtype=float32),
 Array([-2.4424558 , -2.0356805 ,  0.20554423], dtype=float32),
 Array([ 1.4666541, -0.7646161], dtype=float32),
 Array([ 1.4666541, -0.7646161], dtype=float32),
 Array([-25.643421  ,  -0.52879375], dtype=float32))

2. With batch

In [6]:
# batch size
batch_size = 10**4

# initialize input
key1, key2 = jax.random.split(key, 2)
x = jax.random.normal(key1, (batch_size, n))
v = jax.random.normal(key2, (batch_size, n))

# compute jvp
primals, tangents = jax.vmap(jax.jvp, in_axes=(None, 0, 0))(f, (x,), (v,))

primals.shape, tangents.shape

((10000, 2), (10000, 2))

# Compute Vector-jacobian products

1. Without batch

In [7]:
# initialize input
key1, key2 = jax.random.split(key, 2)
x = jax.random.normal(key1, (n,))
u = jax.random.normal(key2, (m,))

# compute vjp
y, vjp_fn = jax.vjp(f, x)

# pull back the covector `u` along `f` evaluated at `x`
v = vjp_fn(u)

#y.shape[0] == m
x, u, f(x), y, v

(Array([ 1.0040143, -0.9063372, -0.7481722], dtype=float32),
 Array([-2.4424558, -2.0356805], dtype=float32),
 Array([ 1.4666541, -0.7646161], dtype=float32),
 Array([ 1.4666541, -0.7646161], dtype=float32),
 (Array([-21.168312 ,  -6.144745 ,  -2.4424558], dtype=float32),))

2. With batch

In [9]:
# initialize input
key1, key2 = jax.random.split(key, 2)
x = jax.random.normal(key1, (batch_size, n))
u = jax.random.normal(key2, (batch_size, m))

# compute vjp
y, vjp_fun = jax.vmap(jax.vjp, in_axes=(None, 0))(f, x)

# pull back the covector `u` along `f` evaluated at `x`
v = jax.vmap(vjp_fun)(u)[0]

y.shape[1] == m, y.shape, v.shape

(True, (10000, 2), (10000, 3))